In [10]:
# ============================================================
# TITANIC - FINAL MODEL & KAGGLE SUBMISSION
# ============================================================

# 1. Imports
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier


# ============================================================
# 2. Load datasets
# ============================================================

train_df = pd.read_csv("D:/My_project/Titanic/Data/Raw/train.csv")
test_df = pd.read_csv("D:/My_project/Titanic/Data/Raw/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (891, 12)
Test shape: (418, 11)


In [11]:
# ============================================================
# 3. Feature Engineering
# ============================================================

def create_features(df):
    df = df.copy()

    # Family size
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

    # Is passenger travelling alone?
    df["isAlone"] = (df["FamilySize"] == 1).astype(int)

    # Age groups
    df["AgeGroup"] = pd.cut(
        df["Age"],
        bins=[0, 12, 18, 30, 50, 80],
        labels=[
            "Child",
            "Teen",
            "YoungAdult",
            "Adult",
            "Senior"
        ]
    )

    # Extract title from name
    df["title"] = (
        df["Name"]
        .str.extract(r",\s*([^.]*)\.")[0]
        .str.strip()
    )

    # Keep common titles, group the rest as Rare
    common_titles = [
        "Mr",
        "Miss",
        "Mrs",
        "Master"
    ]

    df["title"] = df["title"].where(
        df["title"].isin(common_titles),
        "Rare"
    )

    return df

In [12]:
# Apply the SAME feature engineering to train and test

train_df = create_features(train_df)
test_df = create_features(test_df)

print(train_df[[
    "Name",
    "FamilySize",
    "isAlone",
    "AgeGroup",
    "title"
]].head())

                                                Name  FamilySize  isAlone  \
0                            Braund, Mr. Owen Harris           2        0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...           2        0   
2                             Heikkinen, Miss. Laina           1        1   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)           2        0   
4                           Allen, Mr. William Henry           1        1   

     AgeGroup title  
0  YoungAdult    Mr  
1       Adult   Mrs  
2  YoungAdult  Miss  
3       Adult   Mrs  
4       Adult    Mr  


In [13]:
# ============================================================
# 4. Final Features
# ============================================================

features = [
    "Pclass",
    "Sex",
    "AgeGroup",
    "Fare",
    "Embarked",
    "FamilySize",
    "isAlone",
    "title"
]

numeric_features = [
    "Pclass",
    "Fare",
    "FamilySize",
    "isAlone"
]

categorical_features = [
    "Sex",
    "AgeGroup",
    "Embarked",
    "title"
]

X_train = train_df[features]
y_train = train_df["Survived"]

X_test = test_df[features]

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

Training features: (891, 8)
Test features: (418, 8)


In [14]:
# ============================================================
# 5. Preprocessing
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [15]:
# ============================================================
# 6. Final XGBoost Model
# ============================================================

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

In [16]:
# ============================================================
# 7. Train Final Model
# ============================================================

final_model.fit(X_train, y_train)

print("Final model trained successfully.")

Final model trained successfully.


In [17]:
# ============================================================
# 8. Make Predictions
# ============================================================

test_predictions = final_model.predict(X_test)

print("Number of predictions:", len(test_predictions))
print("Prediction counts:")
print(pd.Series(test_predictions).value_counts())

Number of predictions: 418
Prediction counts:
0    265
1    153
Name: count, dtype: int64


In [18]:
# ============================================================
# 9. Create Kaggle Submission
# ============================================================

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_predictions.astype(int)
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")

submission.csv created successfully!


In [19]:
# ============================================================
# 9. Create Kaggle Submission
# ============================================================

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_predictions.astype(int)
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")

submission.csv created successfully!


In [20]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Calculate CV accuracy
scores = cross_val_score(
    final_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("Fold accuracies:", scores)
print(f"Mean CV Accuracy: {scores.mean() * 100:.2f}%")
print(f"Standard Deviation: {scores.std() * 100:.2f}%")

Fold accuracies: [0.83798883 0.85955056 0.81460674 0.83707865 0.84269663]
Mean CV Accuracy: 83.84%
Standard Deviation: 1.44%


In [21]:
print("=" * 40)
print("FINAL MODEL PERFORMANCE")
print("=" * 40)

print(f"Mean CV Accuracy : {scores.mean() * 100:.2f}%")
print(f"CV Std Deviation : {scores.std() * 100:.2f}%")
print(f"Best Fold        : {scores.max() * 100:.2f}%")
print(f"Worst Fold       : {scores.min() * 100:.2f}%")

FINAL MODEL PERFORMANCE
Mean CV Accuracy : 83.84%
CV Std Deviation : 1.44%
Best Fold        : 85.96%
Worst Fold       : 81.46%
